# Using the PhenoMe pipeline with STED-FM models and data
---

## 1. Setup

In [1]:
import torch
import numpy as np
import pandas as pd
from stedfm import get_pretrained_model_v2
from phenome.utils import ModelWrapper
from phenome import PhenoMe, make_dataframe_metadata_fn, load_dinov2_model

# Initialize PhenoMe object
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pheno = PhenoMe(device=device, seed=42)

# Wrap the STED-FM MAE model
class STEDFMWrapper(ModelWrapper):
    def __init__(self, device):
        # Load the pretrained STED-FM model
        model, cfg = get_pretrained_model_v2(
            name = "mae-lightning-small",
            weights = "MAE_SMALL_STED", 
            in_channels=1, # For single channel images
            as_classifier=True,
            # global_pool = "patch" # Returns the embeddings for all patches
        )

        # Initialize the parent class which moves the model to the device and sets it to eval()
        super().__init__(model=model, device=device)

    def _get_embeddings(self, tensor: torch.Tensor) -> torch.Tensor:
        out = self.model.forward_features(tensor)
        return out

stedfm_wrapper = STEDFMWrapper(device=device)

# DinoV2 wrapper for comparison
dinov2_wrapper = load_dinov2_model(model_name="dinov2_vits14_reg", device=device)

in_channels 1
--- mae-lightning-small | (https://s3.valeria.science/flclab-foundation-models/models/mae-small-sted.zip) Downloading from URL and extracting zip ---

Loading state_dict from /Users/renaud/.stedfm/baselines/mae-small_STED/pl_checkpoint-999.pth
--- Loaded model mae-lightning-small with weights MAE_SMALL_STED ---
--- Added linear probe to all frozen blocks ---


Using cache found in /Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main
/Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [35]:
from typing import Any

from sklearn.metrics import silhouette_score, adjusted_rand_score, v_measure_score
from phenome.core import filter_indices

def find_data(pheno: PhenoMe, images_dir: str, masks_dir: str=None, metadata_dir: str=None):
    """ Runs find_files in the given directory
    """
    if metadata_dir:
        metadata_df = pd.read_csv(metadata_dir)
        metadata_fn = make_dataframe_metadata_fn(metadata_df)
    else:
        metadata_fn = None

    file_df = pheno.find_files(images_dir, mask_dir=masks_dir, metadata_fn=metadata_fn)
    return file_df

def group_embeddings(
    pheno: PhenoMe,
    group_by: str,
    exclude: dict[str, Any | list[Any]] | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """ From PhenoMe embeddings and metadata, return numpy arrays of embeddings and their labels for a given metadata column

    Args:
        pheno: Pipeline holding the loaded results.
        group_by: Metadata key used to label each embedding.
        exclude: Optional metadata exclusions, same format as `plot_tsne`'s
            `exclude` argument: maps a metadata key to a single value or a list
            of values to drop, e.g. `{"condition": "Block"}`. Rows matching any
            exclusion are removed from the output.

    Returns:
        Tuple of the (n_kept, D) embedding array and the (n_kept,) label array.
    """
    indices = filter_indices(pheno.results, exclude=exclude)
    emb = pheno.get_embeddings(indices)  # (n_kept, D), row i aligned with indices[i]
    labels = np.array(
        [pheno.results.metadata_value(i, group_by) for i in indices],
        dtype=object,
    )

    return emb, labels

def evaluate_embeddings(pheno: PhenoMe, group_by: str, exclude: dict[str:str] | None=None, n_clusters: int=4) -> None:
    """ Measure various metrics on the PhenoMe embeddings.
    """
    emb, labels = group_embeddings(pheno, group_by=group_by, exclude=exclude)

    # silhouette score
    print("Group silhouette score: ", silhouette_score(emb, labels))

    # clustering
    pheno.compute_clustering(n_clusters=n_clusters)
    emb, clusters_labels = group_embeddings(pheno, group_by="cluster", exclude=exclude)
    print("Clusters silhouette score: ", silhouette_score(emb, clusters_labels))

    # clustering "accuracy": ARI
    print("Adjusted Rand Index: ", adjusted_rand_score(labels, clusters_labels))

    print("\n")

## 2. Neural Activity States Dataset

This dataset contains images of STED images of PSD-95 synaptic protein clusters, under 4 different experimental conditions, labeled Block, 0MgGlyBic, 48hTTX, and GluGly.

In [33]:
# Find image and metadata files
images_dir = "/Users/renaud/datasets/synaptic_proteins/NAS_PSD95_test_v2/images"
metadata_dir = "/Users/renaud/datasets/synaptic_proteins/NAS_PSD95_test_v2/metadata.csv"
file_df = find_data(pheno, images_dir, metadata_dir=metadata_dir)

print(f"Found {len(file_df)} images")
print(f"Metadata keys: {file_df.columns.tolist()}")


Found 1510 files total.
DataFrame: 1510 files, 7 columns.


Found 1510 images
Metadata keys: ['file_path', 'label', 'condition', 'dataset-idx', 'filename', 'id', 'mask_path']


In [37]:

# Results with STED-FM model
pheno.load_results("checkpoints/psd95_dataset_whole/psd95_dataset_stedfm.h5")
evaluate_embeddings(pheno, group_by="condition", n_clusters=4)
fig=pheno.plot_tsne(color_by="condition", return_fig=True, render_mode="svg")
fig.write_image("plots/NAS_dataset_stedfm.svg")

# Results with DinoV2 model
pheno.load_results("checkpoints/psd95_dataset_whole/psd95_dataset_dinov2.h5")
evaluate_embeddings(pheno, group_by="condition", n_clusters=4)
fig=pheno.plot_tsne(color_by="condition", return_fig=True, render_mode="svg")
fig.write_image("plots/NAS_dataset_dinov2.svg")

# Results with STED-FM model - Exclude Block condition
pheno.load_results("checkpoints/psd95_dataset_whole/psd95_dataset_stedfm.h5")
evaluate_embeddings(pheno, group_by="condition", n_clusters=3, exclude={"condition":"Block"})
fig=pheno.plot_tsne(color_by="condition", return_fig=True, render_mode="svg", exclude={"condition":"Block"})
fig.write_image("plots/NAS_dataset_noblock_stedfm.svg")

# Results with DinoV2 model - Exclude Block condition
pheno.load_results("checkpoints/psd95_dataset_whole/psd95_dataset_dinov2.h5")
evaluate_embeddings(pheno, group_by="condition", n_clusters=3, exclude={"condition":"Block"})
fig=pheno.plot_tsne(color_by="condition", return_fig=True, render_mode="svg", exclude={"condition":"Block"})
fig.write_image("plots/NAS_dataset__noblock_dinov2.svg")

Pipeline ready: 1510 images available (lazy loading from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/psd95_dataset_whole/psd95_dataset_stedfm.h5).
Loaded 1510 images from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/psd95_dataset_whole/psd95_dataset_stedfm.h5 (hdf5), 384-dim embeddings (lazy, on disk).
Content: 6 metadata keys (condition, dataset-idx, filename, id, label...), 6 properties (intensity_entropy, intensity_max, intensity_mean, intensity_min, intensity_std...).
Clustering: PCA extracted 100 components from 384 dimensions before k-means.
Clustering (kmeans): 4 clusters on 1510 images (source=embeddings).


Group silhouette score:  0.018813183531165123
Clusters silhouette score:  0.18873517215251923
Adjusted Rand Index:  0.055776322790201005




Pipeline ready: 1510 images available (lazy loading from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/psd95_dataset_whole/psd95_dataset_dinov2.h5).
Loaded 1510 images from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/psd95_dataset_whole/psd95_dataset_dinov2.h5 (hdf5), 384-dim embeddings (lazy, on disk).
Content: 6 metadata keys (condition, dataset-idx, filename, id, label...), 0 properties ().
Clustering: PCA extracted 100 components from 384 dimensions before k-means.
Clustering (kmeans): 4 clusters on 1510 images (source=embeddings).


Group silhouette score:  0.00607020640745759
Clusters silhouette score:  0.1609734296798706
Adjusted Rand Index:  0.1117925884531327




Pipeline ready: 1510 images available (lazy loading from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/psd95_dataset_whole/psd95_dataset_stedfm.h5).
Loaded 1510 images from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/psd95_dataset_whole/psd95_dataset_stedfm.h5 (hdf5), 384-dim embeddings (lazy, on disk).
Content: 6 metadata keys (condition, dataset-idx, filename, id, label...), 6 properties (intensity_entropy, intensity_max, intensity_mean, intensity_min, intensity_std...).
Clustering: PCA extracted 100 components from 384 dimensions before k-means.
Clustering (kmeans): 3 clusters on 1510 images (source=embeddings).


Group silhouette score:  0.10124743729829788
Clusters silhouette score:  0.24573303759098053
Adjusted Rand Index:  0.13384298297444325




Pipeline ready: 1510 images available (lazy loading from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/psd95_dataset_whole/psd95_dataset_dinov2.h5).
Loaded 1510 images from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/psd95_dataset_whole/psd95_dataset_dinov2.h5 (hdf5), 384-dim embeddings (lazy, on disk).
Content: 6 metadata keys (condition, dataset-idx, filename, id, label...), 0 properties ().
Clustering: PCA extracted 100 components from 384 dimensions before k-means.
Clustering (kmeans): 3 clusters on 1510 images (source=embeddings).


Group silhouette score:  0.07030490040779114
Clusters silhouette score:  0.1906566172838211
Adjusted Rand Index:  0.2009202425328669




## 3. Zooniverse Dataset

This dataset contains crops of individual synaptic protein clusters, and their segmentation masks obtained from the Synaptic Protein Zoo citizen science project.

In [39]:
# Find files
from phenome import make_dataframe_metadata_fn
from data_handling import minmax_preprocessing_fn
import pandas as pd

images_dir = "/Users/renaud/datasets/synaptic_proteins/clusters_segmentation_dataset/images"
masks_dir = "/Users/renaud/datasets/synaptic_proteins/clusters_segmentation_dataset/masks"
metadata_dir = "/Users/renaud/datasets/synaptic_proteins/clusters_segmentation_dataset/metadata.csv"

file_df = find_data(pheno, images_dir, metadata_dir=metadata_dir)

print(f"Found {len(file_df)} images")
print(f"Metadata keys: {file_df.columns.tolist()}")

Found 1449 files total.
DataFrame: 1449 files, 33 columns.


Found 1449 images
Metadata keys: ['file_path', 'filename', 'image_name', 'channel', 'protein', 'partner_protein', 'antibody_label', 'condition', 'replicate', 'synapse_class', 'class_index', 'centroid_row', 'centroid_col', 'bbox_row_min', 'bbox_col_min', 'bbox_row_max', 'bbox_col_max', 'area', 'n_components', 'crop_height', 'crop_width', 'content_height', 'content_width', 'min_distance', 'bbox_padding', 'touches_image_border', 'cluster_id', 'class_majority', 'seg_majority', 'class_agreement', 'n_class_votes', 'id', 'mask_path']


In [40]:
# COLOR BY PROTEIN
# Results with STED-FM model
pheno.load_results("checkpoints/segmentation_dataset_clusters/clusters_stretch_stedfm.h5")
evaluate_embeddings(pheno, group_by="protein", n_clusters=4)
fig=pheno.plot_tsne(color_by="protein", return_fig=True, render_mode="svg")
fig.write_image("plots/zoo_dataset_protein_stedfm.svg")

# Results with DinoV2 model
pheno.load_results("checkpoints/segmentation_dataset_clusters/clusters_stretch_dinov2.h5")
evaluate_embeddings(pheno, group_by="protein", n_clusters=4)
fig=pheno.plot_tsne(color_by="protein", return_fig=True, render_mode="svg")
fig.write_image("plots/zoo_dataset_protein_dinov2.svg")


# COLOR BY SYNAPSE CLASS

# Results with STED-FM model
pheno.load_results("checkpoints/segmentation_dataset_clusters/clusters_stretch_stedfm.h5")
fig=pheno.plot_tsne(color_by="synapse_class", return_fig=True, render_mode="svg")
fig.write_image("plots/zoo_dataset_class_stedfm.svg")

# Results with DinoV2 model
pheno.load_results("checkpoints/segmentation_dataset_clusters/clusters_stretch_dinov2.h5")
fig=pheno.plot_tsne(color_by="synapse_class", return_fig=True, render_mode="svg")
fig.write_image("plots/zoo_dataset_class_dinov2.svg")

Pipeline ready: 1449 images available (lazy loading from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/segmentation_dataset_clusters/clusters_stretch_stedfm.h5).
NaNs detected in property columns (NaN count per column): {'ring_1_mean': 2, 'ring_1_std': 2}. 2/1449 images have at least one such NaN. Use detect_nan_properties() for per-column row indices.
Loaded 1449 images from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/segmentation_dataset_clusters/clusters_stretch_stedfm.h5 (hdf5), 384-dim embeddings (lazy, on disk).
Content: 32 metadata keys (antibody_label, area, bbox_col_max, bbox_col_min, bbox_padding...), 32 properties (area, circularity, eccentricity, equivalent_diameter, euler_number...).
Clustering: PCA extracted 100 components from 384 dimensions before k-means.
Clustering (kmeans): 4 clusters on 1449 images (source=embeddings).


Group silhouette score:  0.06917687505483627
Clusters silhouette score:  0.24165983498096466
Adjusted Rand Index:  0.27094236350106604




Pipeline ready: 1449 images available (lazy loading from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/segmentation_dataset_clusters/clusters_stretch_dinov2.h5).
NaNs detected in property columns (NaN count per column): {'ring_1_mean': 2, 'ring_1_std': 2}. 2/1449 images have at least one such NaN. Use detect_nan_properties() for per-column row indices.
Loaded 1449 images from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/segmentation_dataset_clusters/clusters_stretch_dinov2.h5 (hdf5), 384-dim embeddings (lazy, on disk).
Content: 32 metadata keys (antibody_label, area, bbox_col_max, bbox_col_min, bbox_padding...), 32 properties (area, circularity, eccentricity, equivalent_diameter, euler_number...).
Clustering: PCA extracted 100 components from 384 dimensions before k-means.
Clustering (kmeans): 4 clusters on 1449 images (source=embeddings).


Group silhouette score:  0.03097561001777649
Clusters silhouette score:  0.17140449583530426
Adjusted Rand Index:  0.1313704473783668




Pipeline ready: 1449 images available (lazy loading from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/segmentation_dataset_clusters/clusters_stretch_stedfm.h5).
NaNs detected in property columns (NaN count per column): {'ring_1_mean': 2, 'ring_1_std': 2}. 2/1449 images have at least one such NaN. Use detect_nan_properties() for per-column row indices.
Loaded 1449 images from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/segmentation_dataset_clusters/clusters_stretch_stedfm.h5 (hdf5), 384-dim embeddings (lazy, on disk).
Content: 32 metadata keys (antibody_label, area, bbox_col_max, bbox_col_min, bbox_padding...), 32 properties (area, circularity, eccentricity, equivalent_diameter, euler_number...).
Pipeline ready: 1449 images available (lazy loading from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/segmentation_dataset_clusters/clusters_stretch_dinov2.h5).
NaNs detected in

## 4. Optim Dataset

This dataset contains 224x224 crops of STED images of 4 different labeled proteins: 'actin', 'tubulin', 'CaMKII' and 'PSD95'

In [42]:
# Find files
from phenome import make_dataframe_metadata_fn
from data_handling import minmax_preprocessing_fn
import pandas as pd

images_dir = "/Users/renaud/datasets/optim-dataset/images"
metadata_dir = "/Users/renaud/datasets/optim-dataset/metadata.csv"

file_df = find_data(pheno, images_dir, metadata_dir=metadata_dir)

print(f"Found {len(file_df)} images")

Found 438 files total.
DataFrame: 438 files, 7 columns.


Found 438 images


In [43]:
# Results with STED-FM model
pheno.load_results("checkpoints/optim_dataset/optim_stedfm.h5")
evaluate_embeddings(pheno, group_by="label", n_clusters=4)
fig=pheno.plot_tsne(color_by="label", return_fig=True, render_mode="svg")
fig.write_image("plots/optim_dataset_stedfm.svg")

# Results with DinoV2 model
pheno.load_results("checkpoints/optim_dataset/optim_dinov2.h5")
evaluate_embeddings(pheno, group_by="label", n_clusters=4)
fig=pheno.plot_tsne(color_by="label", return_fig=True, render_mode="svg")
fig.write_image("plots/optim_dataset_dinov2.svg")

Pipeline ready: 438 images available (lazy loading from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/optim_dataset/optim_stedfm.h5).
Loaded 438 images from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/optim_dataset/optim_stedfm.h5 (hdf5), 384-dim embeddings (lazy, on disk).
Content: 6 metadata keys (dataset-idx, filename, id, label, mask_path...), 6 properties (intensity_entropy, intensity_max, intensity_mean, intensity_min, intensity_std...).
Clustering: PCA extracted 100 components from 384 dimensions before k-means.
Clustering (kmeans): 4 clusters on 438 images (source=embeddings).


Group silhouette score:  0.14198951423168182
Clusters silhouette score:  0.1958710253238678
Adjusted Rand Index:  0.24946272926837065




Pipeline ready: 438 images available (lazy loading from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/optim_dataset/optim_dinov2.h5).
Loaded 438 images from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/optim_dataset/optim_dinov2.h5 (hdf5), 384-dim embeddings (lazy, on disk).
Content: 6 metadata keys (dataset-idx, filename, id, label, mask_path...), 0 properties ().
Clustering: PCA extracted 100 components from 384 dimensions before k-means.
Clustering (kmeans): 4 clusters on 438 images (source=embeddings).


Group silhouette score:  0.0888998806476593
Clusters silhouette score:  0.1149861216545105
Adjusted Rand Index:  0.2707907953024332


